![image.png](https://i.imgur.com/a3uAqnb.png)
# Lab 3: Transformers — 2017 vs 2026

This notebook builds intuition for the **Transformer architecture** and its modern extensions: scaled dot-product attention, multi-head attention, RoPE, and KV caching.

You will implement core mechanisms from scratch in PyTorch, then relate them to today's large language models.

> 💡 The 2017 Transformer paper is small; 2026 systems reuse the same attention core with different position schemes, caches, and scaling tricks.

__Ready to code attention from first principles?__ Start with installation.



# 📦 Installing Required Python Libraries

This cell installs packages needed for this lab.

- **PyTorch** — Attention, RoPE, and KV-cache implementations.
- **Matplotlib / NumPy** — Optional weight visualization.
- **Transformers** — Reference implementations for sanity checks.


In [1]:
!pip install -q torch torchaudio transformers datasets accelerate matplotlib numpy scipy


# 📥 Importing Essential Python Libraries

Import PyTorch modules used throughout the attention exercises.


In [2]:
import math
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
print(f"PyTorch {torch.__version__}")


PyTorch 2.11.0+cpu


__Theory first — connect definitions to the lecture slides, then move to code.__

---

## 🧠 Part A — Theory

*Reference: baseline 2017 Transformer, positional encodings, RoPE, KV cache, MoE, multimodality.*

### 📖 A1. Core mechanisms

1. Why did Transformers replace RNNs for sequence modeling at scale?
2. Write the scaled dot-product attention formula and explain the role of $\sqrt{d_k}$.
3. What is the purpose of **masked** self-attention in the decoder?

*Write below:*


#### ✍️ Your answers (A1):

1. Transformers enable **parallel training** over sequence length, **direct long-range dependencies** via attention (no vanishing path through time), and **better scaling** with data/compute vs RNNs.

2. $\mathrm{Attention}(Q,K,V) = \mathrm{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right)V$. Scaling by $\sqrt{d_k}$ **prevents dot products from growing large** with dimension, keeping softmax gradients stable.

3. **Masked** self-attention in the decoder prevents positions from attending to **future tokens**, preserving **causal autoregressive** generation during training and inference.

### 📖 A2. 2017 → 2026 evolution

1. Contrast **absolute sinusoidal** positions vs **RoPE** (Rotary Position Embedding).
2. What is stored in the **KV cache** during autoregressive inference, and how does it reduce cost?
3. In a **Mixture-of-Experts (MoE)** layer, what problem does sparse routing address?

*Write below:*


#### ✍️ Your answers (A2):

1. **Sinusoidal absolute** positions add fixed vectors per index; **RoPE** encodes relative position by **rotating** query/key pairs, extrapolating better to **longer contexts**.

2. The **KV cache** stores past **keys and values** for all layers so each new token only computes QKV for itself and attends to cached history — cost per step grows linearly with context but avoids recomputing full sequence attention.

3. **MoE sparse routing** increases **model capacity** (many experts) while activating only a **subset per token**, improving **compute efficiency** vs dense FFNs at same parameter count.

---

## 💻 Part B — Programming
__Let's implement the core ideas in PyTorch.__



### 🛠️ B1. Scaled dot-product attention

In [3]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    """Q,K,V: (batch, heads, seq, d_k). Return (output, weights)."""
    d_k = Q.size(-1)
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(mask, float("-inf"))
    weights = F.softmax(scores, dim=-1)
    output = torch.matmul(weights, V)
    return output, weights

B, H, T, D = 2, 4, 8, 16
Q = torch.randn(B, H, T, D)
K = torch.randn(B, H, T, D)
V = torch.randn(B, H, T, D)
causal = torch.triu(torch.ones(T, T), diagonal=1).bool()
out, w = scaled_dot_product_attention(Q, K, V, mask=causal)
print("Output shape:", out.shape, "| Weights shape:", w.shape)

Output shape: torch.Size([2, 4, 8, 16]) | Weights shape: torch.Size([2, 4, 8, 8])


### 🛠️ B2. Multi-head attention module

In [4]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)

    def forward(self, x, mask=None):
        B, T, _ = x.shape
        Q = self.q_proj(x).view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        K = self.k_proj(x).view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        V = self.v_proj(x).view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        if mask is not None and mask.dim() == 2:
            mask = mask.unsqueeze(0).unsqueeze(0)
        out, _ = scaled_dot_product_attention(Q, K, V, mask=mask)
        out = out.transpose(1, 2).contiguous().view(B, T, -1)
        return self.out_proj(out)

mha = MultiHeadAttention(64, 4)
x = torch.randn(2, 8, 64)
y = mha(x, mask=causal)
print("MHA output shape:", y.shape)

MHA output shape: torch.Size([2, 8, 64])


### 🛠️ B3. RoPE demo

Implement a simplified **rotary** transform on the first two dimensions of each head.

In [5]:
def apply_rope(x: torch.Tensor, positions: torch.Tensor, theta: float = 10000.0) -> torch.Tensor:
    """x: (batch, heads, seq, d_k); positions: (seq,) integer positions."""
    out = x.clone()
    d_k = x.size(-1)
    half = d_k // 2
    for i in range(half):
        freq = 1.0 / (theta ** (2 * i / d_k))
        angles = positions.float() * freq
        cos_a, sin_a = torch.cos(angles), torch.sin(angles)
        x1, x2 = out[..., 2 * i], out[..., 2 * i + 1]
        out[..., 2 * i] = x1 * cos_a - x2 * sin_a
        out[..., 2 * i + 1] = x1 * sin_a + x2 * cos_a
    return out

x = torch.randn(1, 2, T, D)
pos = torch.arange(T)
x_rope = apply_rope(x, pos)
print("RoPE output shape:", x_rope.shape)

RoPE output shape: torch.Size([1, 2, 8, 16])


### 🛠️ B4. KV cache step

Simulate one decode step reusing cached keys and values.

In [6]:
class KVCache:
    def __init__(self):
        self.keys = None
        self.values = None

    def update(self, k, v):
        if self.keys is None:
            self.keys, self.values = k, v
        else:
            self.keys = torch.cat([self.keys, k], dim=2)
            self.values = torch.cat([self.values, v], dim=2)

    def get(self):
        return self.keys, self.values

cache = KVCache()
for step in range(3):
    k_step = torch.randn(B, H, 1, D)
    v_step = torch.randn(B, H, 1, D)
    cache.update(k_step, v_step)
keys, values = cache.get()
print("Cached sequence length:", keys.shape[2])

Cached sequence length: 3


---


### 🛠️ B5. Top-k MoE router with load balancing

Implement sparse routing to `k` of `E` expert MLPs and an auxiliary load-balancing loss.

In [7]:
class MoELayer(nn.Module):
    def __init__(self, d_model: int, n_experts: int = 4, top_k: int = 2):
        super().__init__()
        self.top_k = top_k
        self.router = nn.Linear(d_model, n_experts)
        self.experts = nn.ModuleList([
            nn.Sequential(nn.Linear(d_model, d_model), nn.ReLU(), nn.Linear(d_model, d_model))
            for _ in range(n_experts)
        ])

    def forward(self, x):
        logits = self.router(x)
        topk = torch.topk(logits, self.top_k, dim=-1)
        weights = F.softmax(topk.values, dim=-1)
        out = torch.zeros_like(x)
        usage = torch.zeros(logits.size(-1))
        for k in range(self.top_k):
            idx = topk.indices[..., k]
            w = weights[..., k].unsqueeze(-1)
            for e in range(len(self.experts)):
                mask = idx == e
                if mask.any():
                    out[mask] += w[mask] * self.experts[e](x[mask])
                    usage[e] += mask.sum().float()
        load_loss = usage.std() / (usage.mean() + 1e-8)
        return out, load_loss

moe = MoELayer(64, n_experts=4, top_k=2)
x = torch.randn(16, 64)
y, lb_loss = moe(x)
print(f"MoE output shape: {y.shape}, load-balance loss: {lb_loss.item():.4f}")

MoE output shape: torch.Size([16, 64]), load-balance loss: 0.3385


### 🛠️ B6. RMSNorm vs LayerNorm

Implement **RMSNorm** (used in LLaMA) and compare activation statistics to `LayerNorm`.

In [8]:
class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps

    def forward(self, x):
        rms = torch.sqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        return self.weight * x / rms

x = torch.randn(32, 64) * 5
ln = nn.LayerNorm(64)
rn = RMSNorm(64)
print(f"LayerNorm std: {ln(x).std():.3f} | RMSNorm std: {rn(x).std():.3f}")

LayerNorm std: 1.000 | RMSNorm std: 1.000
